# RAG Saturation & Citation Hallucination Analysis
### Investigating how the number of retrieved documents (*K*) affects answer quality and citation reliability in a Retrieval-Augmented Generation system

**Author:** Bagus Cipta Pratama  ·  **Context:** Selected Topics in Intelligent Systems project
**Stack:** FAISS · Sentence-Transformers · Qwen2.5-1.5B-Instruct (4-bit) · Hugging Face Transformers
**Dataset:** IDK-MRC (Indonesian Knowledge Machine Reading Comprehension)

---

## Executive Summary

Retrieval-Augmented Generation (RAG) couples a *retrieval* engine with a Large Language Model (LLM)
to produce answers grounded in reference documents. A common intuition is *"more documents, better
answers"* — yet an overly long context can inject **noise**, degrade performance, and increase
**citation hallucination**.

This notebook builds an end-to-end RAG pipeline and runs a controlled experiment over
$K \in \{1, 3, 5, 10\}$ to answer three research questions:

1. **Saturation** — does answer quality keep improving as *K* grows?
2. **Citation reliability** — how accurately does the model cite the document that actually contains the answer?
3. **Hallucination** — how does context length relate to the emergence of incorrect citations?

### Key Findings

| K  | ROUGE-1 | BLEU   | Citation Accuracy | Hallucination Rate |
|----|---------|--------|-------------------|--------------------|
| 1  | 0.1842  | 0.0431 | 92.73%            | 7.27%              |
| **3**  | **0.2050**  | **0.0607** | **88.89%**            | **11.11%**             |
| 5  | 0.2011  | 0.0565 | 83.75%            | 16.25%             |
| 10 | 0.1040  | 0.0428 | 31.40%            | 68.60%             |

> **Takeaway:** performance is **non-monotonic** in *K*. The sweet spot is **K = 3**; pushing *K*
> to 10 sharply degrades quality and multiplies the hallucination rate — strong evidence of a
> *noise-injection* effect and the **"Lost in the Middle"** phenomenon.

## 1 · Environment Setup

Install dependencies. Designed for **Google Colab (GPU T4)**; expected runtime ~2–3 minutes.

In [ ]:
# Core dependencies: data, retrieval, modeling, evaluation, and visualization.
%pip install -q datasets faiss-cpu sentence-transformers
%pip install -q -U transformers accelerate bitsandbytes
%pip install -q rouge-score bert-score nltk
%pip install -q matplotlib seaborn pandas tqdm openpyxl

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Dependencies installed successfully.")

## 2 · Imports & Experiment Configuration

Every hyperparameter is centralized in a single configuration object, `CFG`, so the experiment
is easy to reproduce and to vary without touching code in many places.

In [ ]:
from __future__ import annotations

import gc
import os
import re
import json
import time
import shutil
import zipfile
import warnings
from dataclasses import dataclass

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

sns.set_theme(style="whitegrid")

# --- Compute environment summary --------------------------------------------
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name}")
    print(f"VRAM          : {props.total_memory / 1e9:.1f} GB")

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    """Central configuration for the entire RAG pipeline."""

    # Models
    embedding_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    llm_model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    bertscore_model: str = "bert-base-multilingual-cased"

    # Retrieval
    k_values: tuple[int, ...] = (1, 3, 5, 10)
    encode_batch_size: int = 64

    # Generation
    max_new_tokens: int = 256
    max_input_length: int = 2048

    # Experiment
    num_samples: int = 100          # number of evaluation samples (raise for more stable estimates)
    random_seed: int = 42

    # Output
    output_dir: str = "rag_results"


CFG = ExperimentConfig()


def set_seed(seed: int) -> None:
    """Set global seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.random_seed)

print("Experiment configuration:")
for k, v in CFG.__dict__.items():
    print(f"  {k:20s}: {v}")

## 3 · Dataset Loading & Exploration — IDK-MRC

**IDK-MRC** (*Indonesian Knowledge Machine Reading Comprehension*) contains Indonesian
context–question–answer triples. Each row holds a single `context` and a list of `qas`;
some questions are deliberately *unanswerable* (`is_impossible = True`), which serves to
test the model's honesty.

In [ ]:
print("Loading the IDK-MRC dataset ...")
ds = load_dataset("rifkiaputri/idk-mrc")
print(ds)

# --- Inspect a single row ---------------------------------------------------
sample = ds["train"][0]
print("\n[ Example row (split=train) ]")
print(f"Context (200 chars): {sample['context'][:200]}...")
print(f"QA pairs in this row: {len(sample['qas'])}")
for i, qa in enumerate(sample["qas"], 1):
    answer = qa["answers"][0]["text"] if qa["answers"] else "(unanswerable)"
    print(f"  QA-{i}: q='{qa['question']}' | impossible={qa['is_impossible']} | a='{answer}'")

# --- Per-split statistics ---------------------------------------------------
print("\n[ Dataset statistics ]")
for split_name in ["train", "validation", "test"]:
    split = ds[split_name]
    total_qa = sum(len(row["qas"]) for row in split)
    answerable = sum(sum(not qa["is_impossible"] for qa in row["qas"]) for row in split)
    print(f"  {split_name:11s}: {len(split):5d} contexts | "
          f"{total_qa:5d} total QA | {answerable:5d} answerable")

## 4 · Preprocessing & Data Preparation

We extract the **answerable** QA pairs (`is_impossible = False`) together with their
ground-truth contexts. The `validation` split is used as the primary evaluation set.

In [ ]:
def extract_qa_pairs(dataset_split, include_impossible: bool = False):
    """Flatten a split into parallel lists of QA pairs.

    Returns
    -------
    contexts, questions, answers, row_indices : list
        Four aligned lists; ``row_indices`` links each QA back to its source row.
    """
    contexts, questions, answers, row_indices = [], [], [], []

    for row_idx, row in enumerate(dataset_split):
        ctx = row["context"]
        for qa in row["qas"]:
            if qa["is_impossible"]:
                if include_impossible:
                    contexts.append(ctx)
                    questions.append(qa["question"])
                    answers.append("")
                    row_indices.append(row_idx)
                continue
            if not qa["answers"]:
                continue
            contexts.append(ctx)
            questions.append(qa["question"])
            answers.append(qa["answers"][0]["text"])
            row_indices.append(row_idx)

    return contexts, questions, answers, row_indices


val_gt_contexts, val_questions, val_answers, val_row_indices = extract_qa_pairs(ds["validation"])
train_gt_contexts, train_questions, train_answers, train_row_indices = extract_qa_pairs(ds["train"])

print(f"Answerable QA — validation: {len(val_questions)} | train: {len(train_questions)}")
print("\n[ Example QA pair (validation) ]")
print(f"  Question : {val_questions[0]}")
print(f"  Answer   : {val_answers[0]}")
print(f"  Context  : {val_gt_contexts[0][:120]}...")

# --- Text-length statistics (in words) --------------------------------------
ctx_lengths = [len(c.split()) for c in val_gt_contexts[:500]]
q_lengths = [len(q.split()) for q in val_questions[:500]]
ans_lengths = [len(a.split()) for a in val_answers[:500]]

print("\n[ Text length — validation, words ]")
print(f"  Context  : mean={np.mean(ctx_lengths):5.1f} | max={max(ctx_lengths)}")
print(f"  Question : mean={np.mean(q_lengths):5.1f} | max={max(q_lengths)}")
print(f"  Answer   : mean={np.mean(ans_lengths):5.1f} | max={max(ans_lengths)}")

## 5 · Retrieval Corpus Construction

The retrieval corpus is built from **all unique contexts** (train + validation + test),
simulating a real knowledge base in which the retriever must find the relevant document
among thousands of candidates.

In [ ]:
print("Building the corpus from all splits ...")

corpus_contexts, seen = [], set()
for split_name in ["train", "validation", "test"]:
    for row in ds[split_name]:
        ctx = row["context"].strip()
        if ctx not in seen:
            seen.add(ctx)
            corpus_contexts.append(ctx)

# Context -> index mapping for quick verification.
ctx_to_idx = {ctx: i for i, ctx in enumerate(corpus_contexts)}

# Coverage check: every GT context should ideally be present in the corpus.
missing = sum(ctx not in ctx_to_idx for ctx in val_gt_contexts)

print(f"Total unique contexts        : {len(corpus_contexts)}")
print(f"GT contexts missing from corpus: {missing} (expected 0)")
print(f"\nExample corpus document [0]: {corpus_contexts[0][:200]}...")

## 6 · Embedding Model (Sentence-Transformer)

`paraphrase-multilingual-MiniLM-L12-v2` is chosen because it supports 50+ languages
(including Indonesian), is lightweight (~420 MB), and produces **384-dimensional** embeddings —
a good balance between semantic quality and speed.

In [ ]:
print(f"Loading embedding model: {CFG.embedding_model}")
embed_model = SentenceTransformer(CFG.embedding_model)

EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension : {EMBED_DIM}")
print(f"Max sequence length : {embed_model.max_seq_length}")

# Sanity check.
demo_emb = embed_model.encode(["Where was Douwes Dekker born?"])
print(f"\nExample encode -> shape={demo_emb.shape} | norm={np.linalg.norm(demo_emb):.4f}")

## 7 · FAISS Index Construction

All documents are encoded and indexed with **`IndexFlatIP`** (inner product).
Because the embeddings are L2-normalized, inner product is equivalent to **cosine similarity**.

In [ ]:
print(f"Encoding {len(corpus_contexts)} documents (batch={CFG.encode_batch_size}, dim={EMBED_DIM}) ...")

corpus_embeddings = embed_model.encode(
    corpus_contexts,
    batch_size=CFG.encode_batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,   # L2-norm -> inner product == cosine similarity
    show_progress_bar=True,
).astype("float32")

print(f"Embedding matrix: {corpus_embeddings.shape}")

faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(corpus_embeddings)
print(f"FAISS index ready: {faiss_index.ntotal} vectors ({type(faiss_index).__name__})")

# Free the embedding matrix, which is no longer needed.
del corpus_embeddings
gc.collect()

## 8 · Retrieval Function

`retrieve()` encodes the query, searches for the Top-*K* most similar documents in the FAISS
index, and returns the documents together with their similarity scores.

In [ ]:
def retrieve(query: str, k: int = 5):
    """Fetch the Top-K documents most relevant to ``query``.

    Returns
    -------
    contexts, scores, indices : list
        Documents sorted by descending cosine similarity, their scores, and their corpus indices.
    """
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, indices = faiss_index.search(q_emb, k)
    contexts = [corpus_contexts[i] for i in indices[0]]
    return contexts, scores[0].tolist(), indices[0].tolist()


# --- Quick test: is the ground-truth document retrieved? --------------------
test_query, test_gt = val_questions[0], val_gt_contexts[0]
print(f"Query        : {test_query}")
print(f"Ground truth : {val_answers[0]}\n")

for k in CFG.k_values[:3]:
    ctxs, scores, _ = retrieve(test_query, k=k)
    found = any(c == test_gt for c in ctxs)
    print(f"K={k}: GT found={found} | scores={[f'{s:.3f}' for s in scores]}")

## 9 · Generator LLM — Qwen2.5-1.5B-Instruct (4-bit)

An instruction-tuned model with 1.5 B parameters, loaded with **4-bit NF4 quantization**
(*double quantization*) via `bitsandbytes` so it fits on a T4 GPU (15 GB VRAM).
An equivalent fallback in case of out-of-memory: `google/gemma-2b-it`.

In [ ]:
print(f"Loading LLM: {CFG.llm_model} (4-bit NF4)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_tokenizer = AutoTokenizer.from_pretrained(CFG.llm_model, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    CFG.llm_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
llm_model.eval()

if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

print(f"Model loaded: {type(llm_model).__name__}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated / "
          f"{torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

## 10 · Prompt Engineering & Answer Generation

The prompt design presents numbered documents (`[Dokumen 1]`, `[Dokumen 2]`, …) and asks the
model to **state the document number** it relied on. These explicit citations are what we later
measure as *citation accuracy*. Decoding is **greedy** (`do_sample=False`) so results are
deterministic and reproducible.

> The prompt itself is in Indonesian to match the IDK-MRC dataset language; the code and
> narrative around it are in English.

In [ ]:
SYSTEM_PROMPT = (
    "Anda adalah asisten cerdas yang menjawab pertanyaan berdasarkan dokumen "
    "referensi yang diberikan. Jawab secara singkat dan tepat dalam Bahasa "
    "Indonesia. Selalu sebutkan nomor dokumen yang Anda gunakan sebagai referensi."
)


def build_rag_prompt(question: str, contexts: list[str]) -> str:
    """Assemble a RAG prompt with numbered documents and a citation-format instruction."""
    doc_block = "".join(f"[Dokumen {i}]\n{ctx}\n\n" for i, ctx in enumerate(contexts, 1))
    return (
        f"Berikut adalah dokumen referensi:\n\n{doc_block}"
        f"Pertanyaan: {question}\n\n"
        "Instruksi:\n"
        "1. Jawab pertanyaan secara singkat dan tepat berdasarkan dokumen di atas.\n"
        "2. Di akhir jawaban, tambahkan format berikut:\n"
        "   [Referensi: Dokumen X]           - jika menggunakan satu dokumen\n"
        "   [Referensi: Dokumen X, Dokumen Y] - jika menggunakan lebih dari satu\n"
        "   [Referensi: Tidak ada]            - jika tidak ada dokumen yang relevan\n\n"
        "Jawaban:"
    )


def generate_answer(question: str, contexts: list[str], max_new_tokens: int | None = None) -> str:
    """Generate an answer with the LLM, grounded in the retrieved documents (greedy decoding)."""
    max_new_tokens = max_new_tokens or CFG.max_new_tokens

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_rag_prompt(question, contexts)},
    ]
    text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(
        [text], return_tensors="pt", truncation=True, max_length=CFG.max_input_length
    ).to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,               # greedy -> deterministic
            repetition_penalty=1.1,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (drop the prompt portion).
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# --- Generation smoke test (K=3) --------------------------------------------
demo_ctxs, _, _ = retrieve(val_questions[0], k=3)
demo_output = generate_answer(val_questions[0], demo_ctxs)
print(f"Question     : {val_questions[0]}")
print(f"Ground truth : {val_answers[0]}")
print(f"\nLLM answer   :\n{demo_output}")

## 11 · Citation Extraction & Accuracy

The model output is parsed to extract the **document numbers** it referenced. Since the LLM
does not always follow the format, `extract_cited_doc_numbers()` tries several patterns in
sequence (from the most structured down to an aggressive fallback).

**Citation Accuracy** = the proportion of cases where the model cites a document that actually
contains the answer. Cases where the ground-truth document was *not* retrieved are excluded from
the citation score (they cannot be judged fairly).

In [ ]:
def extract_cited_doc_numbers(answer_text: str) -> list[int]:
    """Extract the (1-indexed) document numbers cited by the model, tolerant of format variations."""
    cited: set[int] = set()

    patterns = [
        r"\[Referensi:\s*Dokumen\s*(\d+(?:[,\s]+Dokumen\s*\d+)*)\]",  # [Referensi: Dokumen 1, Dokumen 2]
        r"\[Referensi:\s*(\d+(?:[,\s]+\d+)*)\]",                       # [Referensi: 1, 2]
        r"Referensi:\s*Dokumen\s*(\d+(?:[,\s]+(?:Dokumen\s*)?\d+)*)",  # Referensi: Dokumen X (no brackets)
        r"\(Dokumen\s*(\d+)\)",                                        # (Dokumen 1) inline
    ]
    for pat in patterns:
        for match in re.findall(pat, answer_text, re.IGNORECASE):
            cited.update(int(n) for n in re.findall(r"\d+", match))

    # Aggressive fallback: a bare "Dokumen N" mention with no marker.
    if not cited:
        cited.update(int(n) for n in re.findall(r"[Dd]okumen\s*(\d+)", answer_text))

    return sorted(cited)


def compute_citation_accuracy(cited_doc_numbers, ground_truth_context, retrieved_contexts):
    """Score a single citation sample.

    Returns
    -------
    accuracy : int | None
        1 = correct, 0 = incorrect, None = GT not retrieved (cannot be evaluated).
    gt_in_retrieved : bool
    """
    gt_positions = [i for i, ctx in enumerate(retrieved_contexts, 1) if ctx == ground_truth_context]
    gt_in_retrieved = bool(gt_positions)

    if not gt_in_retrieved:
        return None, False
    if not cited_doc_numbers:
        return 0, True
    hit = any(c in gt_positions for c in cited_doc_numbers)
    return int(hit), True


# --- Extraction test over varied formats ------------------------------------
examples = [
    "Dr. Douwes Dekker lahir di Pasuruan. [Referensi: Dokumen 1]",
    "Beliau lahir di Pasuruan, Jawa Timur [Referensi: Dokumen 2, Dokumen 3]",
    "Saya tidak tahu jawabannya. [Referensi: Tidak ada]",
    "Dilahirkan di Pasuruan menurut Dokumen 1.",
]
for ex in examples:
    print(f"  {str(extract_cited_doc_numbers(ex)):12s} <- {ex[:60]}")

## 12 · Evaluation Metrics

| Metric | Measures | Note |
|--------|----------|------|
| **ROUGE-1/2/L** | Lexical n-gram overlap | stemming disabled (better fit for Indonesian) |
| **BLEU** | N-gram precision | method-4 smoothing, since answers are usually short |
| **BERTScore** | Contextual semantic similarity | computed separately in Section 14 (needs GPU) |

In [ ]:
rouge_eval = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,   # no stemming -> better fit for Indonesian
)


def compute_rouge_scores(prediction: str, reference: str) -> dict[str, float]:
    """ROUGE-1/2/L F-measure between prediction and reference."""
    if not prediction.strip() or not reference.strip():
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    scores = rouge_eval.score(reference, prediction)
    return {k: scores[k].fmeasure for k in ("rouge1", "rouge2", "rougeL")}


def compute_bleu_score(prediction: str, reference: str) -> float:
    """BLEU with method-4 smoothing (Chen & Cherry, 2014) for short answers."""
    if not prediction.strip() or not reference.strip():
        return 0.0
    ref_tokens, hyp_tokens = reference.lower().split(), prediction.lower().split()
    if not hyp_tokens:
        return 0.0
    try:
        return float(sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=SmoothingFunction().method4,
        ))
    except Exception:
        return 0.0


# --- Metric smoke test ------------------------------------------------------
pred_ex, ref_ex = "Dr. Douwes Dekker dilahirkan di Pasuruan, Jawa Timur", "Pasuruan, Jawa Timur"
rouge_ex = compute_rouge_scores(pred_ex, ref_ex)
print(f"Prediction: {pred_ex}")
print(f"Reference : {ref_ex}")
print(f"ROUGE-1={rouge_ex['rouge1']:.4f} | ROUGE-2={rouge_ex['rouge2']:.4f} | "
      f"ROUGE-L={rouge_ex['rougeL']:.4f} | BLEU={compute_bleu_score(pred_ex, ref_ex):.4f}")

## 13 · Main Experiment — Loop over $K \in \{1, 3, 5, 10\}$

For each value of *K*, the following pipeline runs on `num_samples` random questions
(fixed seed → identical samples across *K*, making the comparison **fair**):

1. **Retrieve** *K* documents  →  2. **Generate** an answer  →  3. Compute **ROUGE & BLEU**  →  4. Extract citations & score **citation accuracy**.

> On a Colab T4, 100 samples × 4 values of *K* take ~30 minutes. Raise `CFG.num_samples`
> for more stable estimates at the cost of compute time.

In [ ]:
# Fixed random sample from the validation set (shared across all K -> fair comparison).
set_seed(CFG.random_seed)
n_eval = min(CFG.num_samples, len(val_questions))
sample_idx = np.random.choice(len(val_questions), size=n_eval, replace=False)

eval_questions = [val_questions[i] for i in sample_idx]
eval_answers = [val_answers[i] for i in sample_idx]
eval_gt_contexts = [val_gt_contexts[i] for i in sample_idx]

print("Experiment configuration")
print(f"  K values  : {list(CFG.k_values)}")
print(f"  Samples   : {n_eval}")
print(f"  LLM       : {CFG.llm_model}")
print(f"  Embedding : {CFG.embedding_model}")

all_results: dict[int, dict] = {}
start_time = time.time()

for K in CFG.k_values:
    print(f"\n{'=' * 55}\n  K = {K}\n{'=' * 55}")

    k_data = {
        "K": K,
        "questions": [], "ground_truths": [], "generated_answers": [],
        "rouge1": [], "rouge2": [], "rougeL": [], "bleu": [],
        "citation_correct": [],      # 1=correct, 0=incorrect (evaluable samples only)
        "gt_in_retrieved": [],       # whether GT was retrieved
        "cited_docs": [],            # cited document numbers
        "n_citation_evaluable": 0,
    }

    iterator = tqdm(
        zip(eval_questions, eval_answers, eval_gt_contexts),
        total=n_eval, desc=f"K={K}",
    )
    for i, (question, gt_answer, gt_context) in enumerate(iterator):
        try:
            retrieved, _, _ = retrieve(question, k=K)
            generated = generate_answer(question, retrieved)

            rouge = compute_rouge_scores(generated, gt_answer)
            bleu = compute_bleu_score(generated, gt_answer)
            cited = extract_cited_doc_numbers(generated)
            citation, gt_in_ret = compute_citation_accuracy(cited, gt_context, retrieved)
        except Exception as exc:
            print(f"  [warn] sample {i} failed: {exc}")
            generated, bleu, cited, citation, gt_in_ret = "", 0.0, [], None, False
            rouge = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

        k_data["questions"].append(question)
        k_data["ground_truths"].append(gt_answer)
        k_data["generated_answers"].append(generated)
        k_data["rouge1"].append(rouge["rouge1"])
        k_data["rouge2"].append(rouge["rouge2"])
        k_data["rougeL"].append(rouge["rougeL"])
        k_data["bleu"].append(bleu)
        k_data["gt_in_retrieved"].append(gt_in_ret)
        k_data["cited_docs"].append(cited)
        if citation is not None:
            k_data["citation_correct"].append(citation)
            k_data["n_citation_evaluable"] += 1

    all_results[K] = k_data

    cit_acc = np.mean(k_data["citation_correct"]) if k_data["citation_correct"] else float("nan")
    print(f"  ROUGE-1={np.mean(k_data['rouge1']):.4f} | BLEU={np.mean(k_data['bleu']):.4f} | "
          f"GT retrieved={np.mean(k_data['gt_in_retrieved']) * 100:.1f}% | "
          f"Citation acc={cit_acc:.4f} (n={k_data['n_citation_evaluable']})")

print(f"\nExperiment finished in {(time.time() - start_time) / 60:.1f} minutes.")

## 14 · BERTScore (Semantic Similarity)

BERTScore uses contextual representations (`bert-base-multilingual-cased`) and is therefore
more robust to **paraphrase and synonyms** than ROUGE/BLEU. It is computed separately because
it is batched and GPU-bound.

In [ ]:
print(f"Computing BERTScore ({CFG.bertscore_model}) ...")

for K in CFG.k_values:
    preds = all_results[K]["generated_answers"]
    refs = all_results[K]["ground_truths"]

    # Score non-empty pairs only; the rest get 0.0 to keep lengths aligned.
    valid_mask = [bool(p.strip() and r.strip()) for p, r in zip(preds, refs)]
    valid_preds = [p for p, m in zip(preds, valid_mask) if m]
    valid_refs = [r for r, m in zip(refs, valid_mask) if m]

    if valid_preds:
        try:
            _, _, f1 = bert_score_fn(
                valid_preds, valid_refs,
                model_type=CFG.bertscore_model, lang="id",
                verbose=False, batch_size=16,
            )
            f1_scores = iter(f1.tolist())
        except Exception as exc:
            print(f"  [warn] BERTScore K={K}: {exc}")
            f1_scores = iter([0.0] * len(valid_preds))
    else:
        f1_scores = iter([])

    all_results[K]["bertscore_f1"] = [next(f1_scores) if m else 0.0 for m in valid_mask]
    valid_f1 = [s for s, m in zip(all_results[K]["bertscore_f1"], valid_mask) if m]
    print(f"  K={K}: BERTScore F1 (avg) = {np.mean(valid_f1) if valid_f1 else 0.0:.4f}")

print("BERTScore done.")

## 15 · Metric Summary Table

Aggregate every metric per value of *K* into a single `DataFrame`, complemented by a
**delta** analysis across *K* to highlight the saturation point and the hallucination spike.

In [ ]:
summary_rows = []
for K in CFG.k_values:
    res = all_results[K]
    cit_acc = np.mean(res["citation_correct"]) if res["citation_correct"] else 0.0
    summary_rows.append({
        "K": K,
        "ROUGE-1": round(np.mean(res["rouge1"]), 4),
        "ROUGE-2": round(np.mean(res["rouge2"]), 4),
        "ROUGE-L": round(np.mean(res["rougeL"]), 4),
        "BLEU": round(np.mean(res["bleu"]), 4),
        "BERTScore F1": round(np.mean(res.get("bertscore_f1", [0])), 4),
        "Citation Accuracy": round(cit_acc, 4),
        "Hallucination Rate": round(1.0 - cit_acc, 4),
        "GT in Retrieved (%)": round(np.mean(res["gt_in_retrieved"]) * 100, 2),
        "N Citation Evaluable": res["n_citation_evaluable"],
    })

df_summary = pd.DataFrame(summary_rows)

print("=" * 70)
print(f"RAG RESULTS SUMMARY | IDK-MRC | {CFG.llm_model} | N={CFG.num_samples}")
print("=" * 70)
print(df_summary.to_string(index=False))

# --- Deltas across K --------------------------------------------------------
k_vals = df_summary["K"].tolist()
for col in ["ROUGE-1", "Hallucination Rate"]:
    vals = df_summary[col].tolist()
    print(f"\nDelta {col} across K:")
    for i in range(1, len(k_vals)):
        delta = vals[i] - vals[i - 1]
        arrow = "up  " if delta > 0 else "down"
        print(f"  K={k_vals[i - 1]:2d} -> K={k_vals[i]:2d}: {arrow} {delta:+.4f}")

df_summary

## 16 · Results Visualization

A 2×3 panel summarizes every metric trend against *K*: ROUGE/BLEU/BERTScore scores,
citation accuracy & hallucination, and the relationship between retrieval recall and answer quality.

In [ ]:
K_list = df_summary["K"].tolist()
bar_palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(
    "RAG Saturation & Hallucination Analysis\n"
    f"IDK-MRC | {CFG.llm_model} (4-bit) | Retriever: FAISS | N={CFG.num_samples}",
    fontsize=14, fontweight="bold", y=1.01,
)

# (1) ROUGE scores
ax = axes[0, 0]
for col, marker, color in [("ROUGE-1", "o", "#2196F3"), ("ROUGE-2", "s", "#4CAF50"), ("ROUGE-L", "^", "#F44336")]:
    ax.plot(K_list, df_summary[col], marker + "-", label=col, color=color, lw=2.5, ms=9, zorder=3)
best_idx = df_summary["ROUGE-1"].idxmax()
ax.axvline(K_list[best_idx], color="gray", ls="--", alpha=0.5)
for x, y in zip(K_list, df_summary["ROUGE-1"]):
    ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="#2196F3")
ax.set(title="ROUGE Scores vs K", xlabel="K (number of documents)", ylabel="F1 score")
ax.set_xticks(K_list); ax.legend(); ax.grid(alpha=0.3)

# (2) BLEU
ax = axes[0, 1]
bars = ax.bar(K_list, df_summary["BLEU"], color=bar_palette, edgecolor="black", lw=1.2, alpha=0.85, width=1.8)
for bar, val in zip(bars, df_summary["BLEU"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001, f"{val:.4f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set(title="BLEU Score vs K", xlabel="K (number of documents)", ylabel="BLEU")
ax.set_xticks(K_list); ax.grid(alpha=0.3, axis="y")

# (3) BERTScore
ax = axes[0, 2]
ax.plot(K_list, df_summary["BERTScore F1"], "D-", color="#9C27B0", lw=2.5, ms=9, zorder=3)
ax.fill_between(K_list, df_summary["BERTScore F1"], alpha=0.15, color="#9C27B0")
for x, y in zip(K_list, df_summary["BERTScore F1"]):
    ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9, color="#9C27B0")
ax.set(title="BERTScore F1 (Semantic) vs K", xlabel="K (number of documents)", ylabel="BERTScore F1")
ax.set_xticks(K_list); ax.grid(alpha=0.3)

# (4) Citation accuracy
ax = axes[1, 0]
bars = ax.bar(K_list, df_summary["Citation Accuracy"], color=bar_palette, edgecolor="black", lw=1.2, alpha=0.85, width=1.8)
ax.axhline(0.5, color="red", ls="--", alpha=0.7, label="50% threshold")
for bar, val in zip(bars, df_summary["Citation Accuracy"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{val:.3f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set(title="Citation Accuracy vs K", xlabel="K (number of documents)", ylabel="Citation accuracy (0-1)", ylim=(0, 1.05))
ax.set_xticks(K_list); ax.legend(); ax.grid(alpha=0.3, axis="y")

# (5) Hallucination rate
ax = axes[1, 1]
hall_pct = (df_summary["Hallucination Rate"] * 100).tolist()
bars = ax.bar(K_list, hall_pct, color=["#F44336", "#FF9800", "#FFC107", "#8BC34A"],
              edgecolor="black", lw=1.2, alpha=0.85, width=1.8)
for bar, val in zip(bars, hall_pct):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"{val:.1f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set(title="Citation Hallucination Rate vs K", xlabel="K (number of documents)", ylabel="Hallucination rate (%)", ylim=(0, 105))
ax.set_xticks(K_list); ax.grid(alpha=0.3, axis="y")

# (6) Retrieval recall vs answer quality
ax = axes[1, 2]
ax_twin = ax.twinx()
ax.bar(K_list, df_summary["GT in Retrieved (%)"], color="#607D8B", edgecolor="black", alpha=0.6, width=1.8, label="GT retrieved (%)")
ax_twin.plot(K_list, df_summary["ROUGE-1"], "ro-", lw=2, ms=8, label="ROUGE-1 (right)")
for x, y in zip(K_list, df_summary["GT in Retrieved (%)"]):
    ax.text(x, y + 1, f"{y:.0f}%", ha="center", fontsize=9, color="#263238")
ax.set(title="Retrieval Recall vs ROUGE-1 Saturation", xlabel="K (number of documents)", ylabel="GT in retrieved (%)", ylim=(0, 110))
ax_twin.set_ylabel("ROUGE-1", color="red")
ax.set_xticks(K_list); ax.grid(alpha=0.3, axis="y")
lines = ax.get_legend_handles_labels()[0] + ax_twin.get_legend_handles_labels()[0]
labels = ax.get_legend_handles_labels()[1] + ax_twin.get_legend_handles_labels()[1]
ax.legend(lines, labels, fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig("rag_performance_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: rag_performance_analysis.png")

## 17 · Per-*K* Metric Distribution (Box Plot)

Means can hide variance. Box plots with a jittered point overlay reveal the **full spread**
of per-sample scores, making the stability of each *K* configuration clearly visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Metric Distribution per K", fontsize=14, fontweight="bold")

metrics_to_plot = [
    ("rouge1", "ROUGE-1", "#2196F3"),
    ("bleu", "BLEU", "#9C27B0"),
    ("bertscore_f1", "BERTScore F1", "#FF9800"),
]
box_colors = ["#BBDEFB", "#90CAF9", "#64B5F6", "#42A5F5"]

for ax, (metric_key, title, scatter_color) in zip(axes, metrics_to_plot):
    data_per_k = [all_results[K].get(metric_key, []) for K in CFG.k_values]

    bp = ax.boxplot(
        data_per_k, labels=[f"K={k}" for k in CFG.k_values],
        patch_artist=True, medianprops={"color": "red", "linewidth": 2},
    )
    for patch, color in zip(bp["boxes"], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    for i, data in enumerate(data_per_k, 1):
        jitter = np.random.normal(0, 0.05, len(data))
        ax.scatter(i + jitter, data, alpha=0.2, s=8, color=scatter_color, zorder=2)

    means = [np.mean(d) if len(d) else 0.0 for d in data_per_k]
    ax.plot(range(1, len(CFG.k_values) + 1), means, "k--o", ms=5, lw=1.5, label="Mean", zorder=5)
    ax.set(title=f"{title} distribution", ylabel="Score")
    ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("rag_distribution_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: rag_distribution_boxplot.png")

## 18 · Qualitative Analysis

Complements the aggregate metrics with **concrete examples** that highlight three scenarios:
(a) a larger *K* helps, (b) a larger *K* instead injects *noise*, and (c) citation hallucination
(the correct document is retrieved but the model cites the wrong one).

In [ ]:
pd.set_option("display.max_colwidth", 200)


def show_comparison(idx: int, title: str = "") -> None:
    """Display answers for all K values side by side for a single sample."""
    print(f"\n{'=' * 70}\n{title} (sample #{idx})\n{'=' * 70}")
    print(f"Question     : {eval_questions[idx]}")
    print(f"Ground truth : {eval_answers[idx]}")
    for K in CFG.k_values:
        gen = all_results[K]["generated_answers"][idx]
        cited = all_results[K]["cited_docs"][idx]
        r1 = all_results[K]["rouge1"][idx]
        preview = gen[:200] + ("..." if len(gen) > 200 else "")
        print(f"\n  K={K:2d} | ROUGE-1={r1:.3f} | citations={cited}\n    {preview}")


# (a) Larger K helps the most (ROUGE-1 at K=5 >> K=1)
gain = [all_results[5]["rouge1"][i] - all_results[1]["rouge1"][i] for i in range(len(eval_questions))]
show_comparison(int(np.argmax(gain)), "Larger K HELPS (ROUGE-1 rises significantly)")

# (b) Larger K hurts (ROUGE-1 at K=1 >> K=10) -> noise / saturation
loss = [all_results[1]["rouge1"][i] - all_results[10]["rouge1"][i] for i in range(len(eval_questions))]
show_comparison(int(np.argmax(loss)), "Larger K does NOT help (noise / saturation)")

# (c) A citation-hallucination example at K=5
for i in range(len(eval_questions)):
    correct = all_results[5]["citation_correct"]
    if all_results[5]["gt_in_retrieved"][i] and i < len(correct) and correct[i] == 0:
        show_comparison(i, "CITATION HALLUCINATION (GT retrieved, but cited incorrectly)")
        break

## 19 · Saturation & Hallucination Analysis

A quantitative synthesis of the findings: answer-quality trends, retrieval recall, and the
dynamics of citation hallucination, closing with a report-ready conclusion.

In [ ]:
r1 = df_summary["ROUGE-1"].tolist()
ca = df_summary["Citation Accuracy"].tolist()
hr = df_summary["Hallucination Rate"].tolist()
gt = df_summary["GT in Retrieved (%)"].tolist()
k_list = df_summary["K"].tolist()

print("=" * 70)
print("RAG PERFORMANCE SATURATION ANALYSIS")
print("=" * 70)

print("\n1. ANSWER-QUALITY TREND")
for i, K in enumerate(k_list):
    print(f"   K={K:2d}: ROUGE-1={r1[i]:.4f} | BLEU={df_summary['BLEU'][i]:.4f} | "
          f"BERTScore={df_summary['BERTScore F1'][i]:.4f}")
best_k = k_list[int(np.argmax(r1))]
print(f"   -> Best K (ROUGE-1): K={best_k}")
for i in range(1, len(k_list)):
    print(f"   -> Change K={k_list[i-1]}->{k_list[i]}: {r1[i] - r1[i-1]:+.4f}")
if len(r1) >= 3 and (r1[1] - r1[0]) > (r1[2] - r1[1]):
    print("   -> Saturation detected: marginal gains shrink after the initial K.")

print("\n2. RETRIEVAL RECALL")
for i, K in enumerate(k_list):
    print(f"   K={K:2d}: GT in retrieved = {gt[i]:.1f}%")
print("   -> Recall rises with K, but the extra documents are often irrelevant (noise).")

print("\n3. CITATION HALLUCINATION")
for i, K in enumerate(k_list):
    print(f"   K={K:2d}: Citation accuracy={ca[i]:.4f} | Hallucination rate={hr[i] * 100:.1f}% | "
          f"N eval={df_summary['N Citation Evaluable'].iloc[i]}")
print(f"   -> Highest citation accuracy: K={k_list[int(np.argmax(ca))]}")
print("   -> A long context makes it harder for the model to identify the relevant document")
print("      (the 'Lost in the Middle' phenomenon).")

print("\n" + "=" * 70)
print("CONCLUSION")
print("=" * 70)
print(f"  - Quality saturation starts to appear beyond K = {best_k}.")
print("  - Increasing K raises recall, but also adds noise.")
print("  - Citation accuracy drops as K grows: the model struggles to pick the relevant document.")
print("  - Optimal trade-off: choose the K that balances recall and precision.")

## 20 · Result Persistence

All artifacts (summary table, per-sample results, raw JSON dump, and figures) are saved to
the `rag_results/` directory so they can be reproduced and attached to the report.

In [ ]:
os.makedirs(CFG.output_dir, exist_ok=True)

# (1) Summary table
df_summary.to_csv(f"{CFG.output_dir}/summary_table.csv", index=False)

# (2) Detailed per-sample results
detail_rows = []
for K in CFG.k_values:
    res = all_results[K]
    bert = res.get("bertscore_f1", [None] * len(res["questions"]))
    for i in range(len(res["questions"])):
        detail_rows.append({
            "K": K,
            "question": res["questions"][i],
            "ground_truth": res["ground_truths"][i],
            "generated_answer": res["generated_answers"][i],
            "rouge1": res["rouge1"][i],
            "rouge2": res["rouge2"][i],
            "rougeL": res["rougeL"][i],
            "bleu": res["bleu"][i],
            "bertscore_f1": bert[i],
            "cited_docs": str(res["cited_docs"][i]),
            "gt_in_retrieved": res["gt_in_retrieved"][i],
        })
pd.DataFrame(detail_rows).to_csv(f"{CFG.output_dir}/detailed_results.csv", index=False)

# (3) Raw JSON dump (serializable fields only)
serializable = {
    str(K): {k: v for k, v in all_results[K].items() if isinstance(v, (list, int, float, str))}
    for K in CFG.k_values
}
with open(f"{CFG.output_dir}/raw_results.json", "w", encoding="utf-8") as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)

# (4) Copy figures
for img in ("rag_performance_analysis.png", "rag_distribution_boxplot.png"):
    if os.path.exists(img):
        shutil.copy(img, f"{CFG.output_dir}/{img}")

print(f"Artifacts saved to '{CFG.output_dir}/':")
for name in sorted(os.listdir(CFG.output_dir)):
    size_kb = os.path.getsize(os.path.join(CFG.output_dir, name)) / 1024
    print(f"  {name:35s} ({size_kb:6.1f} KB)")

## 21 · Artifact Export

Bundle `rag_results/` into a single ZIP archive. On Google Colab the file download is triggered
automatically; in a local environment the archive is simply written to disk.

In [ ]:
zip_path = "rag_experiment_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(CFG.output_dir):
        for fname in fnames:
            zf.write(os.path.join(root, fname))
print(f"Archive created: {zip_path}")

try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
    print("Download started (Google Colab).")
except ImportError:
    print(f"Non-Colab environment -> artifacts available in '{CFG.output_dir}/' and '{zip_path}'.")
except Exception as exc:
    print(f"Failed to trigger download: {exc}. Artifacts remain in '{CFG.output_dir}/'.")